# Chapter 12 Lab — Extend the Pipeline

The four agents and the orchestrator from the chapter are defined for you in
the **Setup** section below, so you can focus on the additions.

**Your tasks:**
1. Add a `categorize_product` agent (LLM, output constrained to a fixed list).
2. Wire it into `run_one` — only when `category` is missing, and non-fatal.
3. Add `duration_seconds` timing to each `PipelineRecord`.
4. Report: how many needed categorization, average duration, status counts.

Worked answers are in Section 12.9 of the chapter.

## Setup — environment + all four agents (provided)

In [ ]:
import os
from dotenv import load_dotenv
import openai

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")


In [ ]:
# Listing 12.1 The pipeline data contract
from typing import List, Optional

from pydantic import BaseModel, Field


class ProductInput(BaseModel):  #A
    """What goes into the pipeline: brand + product name."""
    brand_name: str
    product_name: str


class ProductExtraction(BaseModel):  #B
    """Raw fields the LLM pulls from a page (from Chapter 11)."""
    product_name: str
    brand_name: str
    description: Optional[str] = None
    price: Optional[str] = None
    weight: Optional[str] = None
    primary_image_url: Optional[str] = None
    category: Optional[str] = None


class EnrichedProduct(BaseModel):  #C
    """The validated, normalized record (pipeline Stage 6)."""
    brand_name: str
    product_name: str
    product_url: Optional[str] = None
    description: Optional[str] = None
    price_usd: Optional[float] = None
    weight_value: Optional[float] = None
    weight_unit: Optional[str] = None
    primary_image_url: Optional[str] = None
    category: Optional[str] = None


class PipelineRecord(BaseModel):  #D
    """The unit of work that flows through every agent."""
    search_key: str
    status: str = "pending"  #E
    stage: str = "start"  #F
    url: Optional[str] = None
    confidence: Optional[str] = None
    enriched: Optional[EnrichedProduct] = None
    notes: List[str] = Field(default_factory=list)  #G
    error: Optional[str] = None

#A The pipeline input: exactly what RuckZone gave us in Chapter 10
#B The unstructured fields an LLM returns; same schema as Chapter 11
#C The clean, typed record we actually want to store
#D One PipelineRecord per product carries state across all stages
#E Lifecycle status: pending, success, needs_review, or failed
#F The last stage that touched this record, for debugging
#G Validation issues collected along the way, for the review queue


In [ ]:
# Listing 12.2 Logging and a reusable retry decorator
import functools
import logging
import sys
import time

logger = logging.getLogger("ruckzone.pipeline")  #A


def configure_logging(level=logging.INFO):  #B
    """Send pipeline logs to stdout with timestamps."""
    logger.setLevel(level)
    if not logger.handlers:  #C
        handler = logging.StreamHandler(sys.stdout)
        handler.setFormatter(
            logging.Formatter(
                "%(asctime)s %(levelname)s | %(message)s",
                datefmt="%H:%M:%S",
            )
        )
        logger.addHandler(handler)
    return logger


def with_retries(max_attempts=3, base_delay=1.0,
                 exceptions=(Exception,)):  #D
    """Retry a function with exponential backoff."""
    def decorator(func):
        @functools.wraps(func)  #E
        def wrapper(*args, **kwargs):
            attempt = 1
            while True:
                try:
                    return func(*args, **kwargs)  #F
                except exceptions as exc:
                    if attempt >= max_attempts:  #G
                        logger.warning(
                            "%s failed after %d attempts: %s",
                            func.__name__, attempt, exc,
                        )
                        raise
                    delay = base_delay * (2 ** (attempt - 1))  #H
                    logger.info(
                        "%s attempt %d failed (%s); retry in %.1fs",
                        func.__name__, attempt, exc, delay,
                    )
                    time.sleep(delay)
                    attempt += 1
        return wrapper
    return decorator

#A One named logger shared by every agent in the pipeline
#B Call once at the start of a run to turn logging on
#C Guard against adding duplicate handlers if the cell re-runs
#D A decorator any agent can wear to become resilient
#E Preserve the wrapped function's name so logs stay readable
#F The happy path: succeed and return immediately
#G Out of attempts, log the failure and re-raise for the caller
#H Wait longer after each failure (1s, 2s, 4s, ...)


In [ ]:
# Listing 12.3 Agent 1: URL discovery
import os

import openai
import requests
from pydantic import BaseModel

SERPAPI_KEY = os.getenv("SERPAPI_KEY")  #A


class URLRanking(BaseModel):  #B
    best_url: str
    confidence: str
    reasoning: str


URL_RANKING_SYSTEM_PROMPT = """
You are a data engineering assistant building a product database.
Given a product search key and candidate URLs, pick the single best
URL for extracting structured product data.
Prefer manufacturer/official pages and individual product pages.
Avoid review sites, forums, and category pages.
Return the best URL, a confidence level (high/medium/low), and a
brief reason.
"""  #C


@with_retries(max_attempts=3, exceptions=(requests.RequestException,))  #D
def _search(search_key, num_results=5):
    params = {
        "q": search_key,
        "api_key": SERPAPI_KEY,
        "num": num_results,
        "engine": "google",
    }
    resp = requests.get(
        "https://serpapi.com/search", params=params, timeout=15
    )
    resp.raise_for_status()
    return [
        {
            "title": r.get("title", ""),
            "url": r.get("link", ""),
            "snippet": r.get("snippet", ""),
        }
        for r in resp.json().get("organic_results", [])
    ]


def discover_url(search_key: str):  #E
    """Agent: find candidate URLs and let an LLM pick the best."""
    candidates = _search(search_key)  #F
    if not candidates:
        return None  #G
    listing = "\n".join(
        f"- {c['title']} | {c['url']} | {c['snippet']}"
        for c in candidates
    )
    user_prompt = (
        f"Product search key: {search_key}\n\n"
        f"Candidate URLs:\n{listing}"
    )
    completion = openai.beta.chat.completions.parse(  #H
        model="gpt-4o-mini",
        messages=[
            {"role": "system",
             "content": URL_RANKING_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format=URLRanking,
    )
    return completion.choices[0].message.parsed  #I

#A Read the SerpAPI key from the environment (see sample.env)
#B The structured ranking contract, identical to Chapter 11
#C Decision rules that tell the model what a good source looks like
#D The search call wears the retry decorator from Listing 12.2
#E The agent: one job, search key in, a ranked URL out
#F Step 1, get candidate URLs from the search API
#G No candidates means nothing to rank; signal failure with None
#H Step 2, ask the cheap model to rank and justify its pick
#I Return the validated URLRanking object for the next agent


In [ ]:
# Listing 12.4 Agent 2: fetch and clean
import requests
from bs4 import BeautifulSoup

REMOVE_TAGS = [  #A
    "script", "style", "nav", "footer", "header",
    "iframe", "noscript", "svg", "form",
]
REMOVE_CLASSES = [
    "breadcrumb", "related-products", "recently-viewed",
    "newsletter", "cookie-banner", "site-footer", "site-header",
    "cart-drawer", "search-modal", "review", "reviews", "ratings",
]

SESSION = requests.Session()  #B
SESSION.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/121.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
})


@with_retries(max_attempts=3, exceptions=(requests.RequestException,))  #C
def _fetch(url):
    resp = SESSION.get(url, timeout=20)
    resp.raise_for_status()
    return resp.text


def clean_html_aggressive(html: str) -> str:  #D
    """Strip non-product noise; returns extraction-ready text."""
    soup = BeautifulSoup(html, "html.parser")
    for tag_name in REMOVE_TAGS:
        for element in soup.find_all(tag_name):
            element.decompose()
    for pattern in REMOVE_CLASSES:
        for element in soup.find_all(
            class_=lambda c: c and pattern in " ".join(c).lower()
        ):
            element.decompose()
    return " ".join(soup.stripped_strings)


def fetch_and_clean(url: str) -> str:  #E
    """Agent: fetch a URL and return cleaned page text."""
    raw_html = _fetch(url)  #F
    cleaned = clean_html_aggressive(raw_html)  #G
    if len(cleaned) < 200:  #H
        raise ValueError("page too small after cleaning")
    return cleaned

#A The tag and class noise lists, carried over from Chapter 11
#B One reused Session with browser-like headers for every fetch
#C Network calls are flaky, so the fetch retries with backoff
#D The aggressive cleaner from Chapter 11, unchanged
#E The agent: a URL in, clean extraction-ready text out
#F Step 1, fetch raw HTML (with retries handled for us)
#G Step 2, strip the noise so we send fewer tokens downstream
#H Guard: a near-empty page usually means a block or JS-only render


In [ ]:
# Listing 12.5 Agent 3: AI extraction
import openai

EXTRACTION_PROMPT = """You are a product data extraction assistant.
Given the text of a product web page, extract these fields:
- product_name, brand_name, description, price, weight,
  primary_image_url, category
Rules:
- Only extract values explicitly present in the text.
- Use null for anything you cannot find; never guess.
- For price, prefer the current/sale price with its symbol.
- For weight, include the unit (lbs, oz, kg, g).
"""  #A


@with_retries(max_attempts=2, exceptions=(openai.OpenAIError,))  #B
def extract_product(cleaned_text: str) -> ProductExtraction:  #C
    """Agent: pull structured fields from cleaned text."""
    response = openai.beta.chat.completions.parse(  #D
        model="gpt-4o",
        messages=[
            {"role": "system", "content": EXTRACTION_PROMPT},
            {"role": "user", "content": cleaned_text[:8000]},  #E
        ],
        response_format=ProductExtraction,  #F
    )
    return response.choices[0].message.parsed  #G

#A The extraction rules from Chapter 11, kept deliberately strict
#B Retry once on transient API errors; extraction is the costly call
#C The agent: clean text in, a ProductExtraction object out
#D Use the more capable model for the field-level reasoning
#E Cap input length to control tokens and cost
#F The Pydantic schema is the contract the model must satisfy
#G Return the parsed object for validation in the next agent


In [ ]:
# Listing 12.6 Agent 4: validate and map (Stage 6)
import re
from typing import List, Optional, Tuple

REQUIRED_FIELDS = ["product_name", "brand_name", "price_usd"]  #A


def _parse_price(price: Optional[str]) -> Optional[float]:  #B
    if not price:
        return None
    match = re.search(r"[\d,]+\.?\d*", price.replace(",", ""))
    return float(match.group()) if match else None


def _parse_weight(weight: Optional[str]
                  ) -> Tuple[Optional[float], Optional[str]]:  #C
    if not weight:
        return None, None
    match = re.search(r"([\d.]+)\s*(lbs?|oz|kg|g)\b", weight, re.I)
    if not match:
        return None, None
    return float(match.group(1)), match.group(2).lower()


def validate_and_map(
    extraction: ProductExtraction,
    url: Optional[str],
    fallback: ProductInput,
) -> Tuple[EnrichedProduct, List[str]]:  #D
    """Agent: normalize values into the final schema and check them."""
    issues: List[str] = []
    price = _parse_price(extraction.price)  #E
    weight_value, weight_unit = _parse_weight(extraction.weight)
    record = EnrichedProduct(  #F
        brand_name=extraction.brand_name or fallback.brand_name,
        product_name=extraction.product_name or fallback.product_name,
        product_url=url,
        description=extraction.description,
        price_usd=price,
        weight_value=weight_value,
        weight_unit=weight_unit,
        primary_image_url=extraction.primary_image_url,
        category=extraction.category,
    )
    for field in REQUIRED_FIELDS:  #G
        if getattr(record, field) in (None, ""):
            issues.append(f"missing {field}")
    if price is not None and not (0 < price < 100000):  #H
        issues.append(f"price out of range: {price}")
    return record, issues

#A The fields a record must have to be trusted without review
#B Turn a messy price string like "$395.00" into a float
#C Split "4.21 lbs" into a numeric value and a unit string
#D The agent: raw extraction in, a clean record plus issue list out
#E Deterministic normalization, no LLM needed here
#F Build the typed record, falling back to the known input fields
#G Flag any required field that came back empty
#H Sanity-check the price; obvious outliers go to review, not the DB


In [ ]:
# Listing 12.7 The orchestrator: chaining the agents
import time
from typing import Dict, List


def run_one(product: ProductInput) -> PipelineRecord:  #A
    """Run a single product through every agent, tracking state."""
    key = f"{product.brand_name} {product.product_name}"
    record = PipelineRecord(search_key=key)
    try:
        record.stage = "discover"  #B
        ranking = discover_url(key)
        if ranking is None:
            record.status = "failed"
            record.error = "no candidate URLs"
            return record
        record.url = ranking.best_url
        record.confidence = ranking.confidence

        record.stage = "fetch"  #C
        cleaned = fetch_and_clean(record.url)

        record.stage = "extract"  #D
        extraction = extract_product(cleaned)

        record.stage = "validate"  #E
        enriched, issues = validate_and_map(
            extraction, record.url, product
        )
        record.enriched = enriched
        record.notes = issues

        if issues or ranking.confidence.lower() == "low":  #F
            record.status = "needs_review"
        else:
            record.status = "success"
    except Exception as exc:  #G
        record.status = "failed"
        record.error = f"{type(exc).__name__}: {exc}"
        logger.warning(
            "%s failed at %s: %s", key, record.stage, exc
        )
    return record


def run_pipeline(products: List[ProductInput]) -> Dict:  #H
    """Run every product and split results from the review queue."""
    results, review_queue = [], []
    for product in products:
        record = run_one(product)  #I
        results.append(record)
        if record.status == "needs_review":
            review_queue.append(record)
        logger.info("%s -> %s", record.search_key, record.status)
        time.sleep(1)  #J
    return {"results": results, "review_queue": review_queue}

#A One product, one record, threaded through the four agents in order
#B Stage 2 (Ch11): discover the best URL, or fail fast if none
#C Stage 3: fetch and clean, with retries inside the agent
#D Stage 5: AI extraction into the ProductExtraction schema
#E Stage 6: normalize and validate into an EnrichedProduct
#F Low confidence or any issue routes the record to human review
#G One catch-all: a failure in any stage is recorded, never fatal
#H The orchestrator over a batch of products
#I Each record is independent, so one failure never stops the batch
#J A short pause keeps us from hammering sites and APIs


## 1. Add a categorization agent
Write `categorize_product(enriched) -> str` that uses an LLM to assign one
category from: backpack, tent, sleeping bag, sleeping pad, stove, water
filter, headlamp, other. Constrain the output with a Pydantic model.

In [ ]:
# Your code here


## 2. Wire it into the orchestrator
In `run_one`, after `validate_and_map`, call your agent **only** when
`enriched.category` is missing. A failure here must not fail the product.

In [ ]:
# Your modified run_one here


## 3. Add timing
Add `duration_seconds` to `PipelineRecord` and measure elapsed time in
`run_one` (a `finally` block guarantees it is always set).

In [ ]:
# Your code here


## 4. Run and report
Run the pipeline on the five chapter products and summarize: how many needed
categorization, the average duration per product, and the status counts.

In [ ]:
products = [
    ProductInput(brand_name=b, product_name=n)
    for b, n in [
        ("GORUCK", "GR1 26L"),
        ("GORUCK", "Rucker 4.0 20L"),
        ("Osprey", "Atmos AG 65"),
        ("Petzl", "Actik Core"),
        ("Sawyer", "Squeeze"),
    ]
]
configure_logging()
# outcome = run_pipeline(products)
# ... build your report DataFrame here
